# Uji Kendali Penglihatan — Skripsi Alif (versi Kaggle)

Smoke test v2 memperlihatkan 19 dari 20 keluaran berisi `tidak_jelas` untuk keenam fitur.
Ada dua penjelasan yang sama masuk akal, dan keduanya menuntut tindakan berbeda:

- **(H1) Prompt terlalu menahan.** Aturan "jangan menebak, jawab tidak_jelas" terlalu kuat,
  sehingga model memilih jalan aman. Kalau ini penyebabnya, prompt tinggal dilonggarkan.
- **(H2) Model memang tidak dapat membaca chip.** Bangunan selebar ~15 m pada 0,5 m/piksel,
  dipotong 128 px lalu diperbesar, mungkin di luar kemampuan model 2–3 miliar parameter.
  Kalau ini penyebabnya, melonggarkan prompt hanya akan menghasilkan tebakan, bukan penglihatan.

Notebook ini memisahkan keduanya. Tidak ada JSON, tidak ada rubrik, tidak ada pilihan
`tidak_jelas`. Hanya pertanyaan telanjang, pada tiga chip yang kondisinya berbeda jauh,
dengan empat cara penyajian.

**Bacaannya sederhana:** kalau jawaban **berbeda-beda antarchip**, model melihat (H1 benar).
Kalau jawaban **seragam untuk chip utuh maupun chip hancur**, model tidak melihat (H2 benar).

### Prasyarat di Kaggle (wajib, sebelum menjalankan apa pun)

Buka panel **Notebook options** di kanan, lalu pastikan tiga hal berikut:

1. **Accelerator** = `GPU T4 x2` atau `GPU P100`. Tanpa ini sel 1 berhenti.
2. **Internet** = `On`. Kaggle mematikan internet secara bawaan, dan tanpa internet
   bobot model tidak dapat diunduh dari Hugging Face. Ini beda terbesar dengan Colab.
3. **Data** > *Add Input* > unggah dataset berisi chip, sampel, dan prompt.

Butir 2 dan 3 memerlukan akun Kaggle yang sudah **terverifikasi nomor telepon**.

Jalankan sel 1 sampai 8 berurutan, lalu kirim `uji_kendali_mentah.jsonl`
(ada di tab **Output** di kanan, atau di `/kaggle/working/`).

## Sel 1 — periksa GPU

In [ ]:
!nvidia-smi

In [ ]:
import sys, torch
print("python :", sys.version.split()[0], "| torch:", torch.__version__,
      "| cuda:", torch.cuda.is_available())
if not torch.cuda.is_available():
    print("\n>>> GPU TIDAK AKTIF. Runtime > Change runtime type > T4 GPU > Save.")
else:
    p = torch.cuda.get_device_properties(0)
    print("gpu    :", p.name, f"| {p.total_memory/1e9:.1f} GB")

## Sel 2 — pasang pustaka

Versi `transformers` tetap dipaku di 4.51.3, sama seperti notebook smoke test.
Kaggle akan mengeluhkan bentrok dependensi dengan paket bawaannya; abaikan, yang
menentukan adalah sel 5 dan 6 berhasil memuat model.

Sel ini juga memeriksa internet, karena itu penyebab kegagalan paling umum di Kaggle.

In [ ]:
!pip install -q "transformers==4.51.3" accelerate timm einops sentencepiece
import transformers, urllib.request
print("transformers:", transformers.__version__)
try:
    urllib.request.urlopen("https://huggingface.co", timeout=10)
    print("internet    : AKTIF")
except Exception as e:
    print("internet    : MATI ->", type(e).__name__)
    print("\n>>> Nyalakan Internet di panel Notebook options (kanan), lalu ulangi sel ini.")
    print(">>> Tanpa internet, bobot model tidak bisa diunduh.")

## Sel 3 — temukan dataset dan pilih tiga chip pembanding

Dipilih satu chip per kelas: `no-damage`, `major-damage`, `destroyed`. Tiga kondisi yang
secara visual berjauhan, supaya kalau model memang melihat, perbedaannya tidak mungkin
terlewat.

In [ ]:
from pathlib import Path
import pandas as pd

# Di Kaggle dataset dipasang read-only di /kaggle/input/<nama-dataset>/,
# dan satu-satunya folder yang boleh ditulisi adalah /kaggle/working/.
AKAR_INPUT = Path("/kaggle/input")
KERJA = Path("/kaggle/working")

terpasang = sorted(p for p in AKAR_INPUT.glob("*") if p.is_dir())
print("dataset terpasang:", [p.name for p in terpasang] or "(belum ada)")
BASIS = next((d for d in terpasang if list(d.rglob("sampel_kecil.csv"))), None)
if BASIS is None:
    raise FileNotFoundError(
        "Tidak ada dataset yang memuat sampel_kecil.csv.\n"
        "Add Input > unggah zip berisi Output Chip, Output Sampel, dan Output Prompt.")
print("dataset dipakai  :", BASIS)

MODEL_QWEN = "Qwen/Qwen2.5-VL-3B-Instruct"
MODEL_INTERNVL = "OpenGVLab/InternVL3-2B"
MAX_NEW_TOKENS = 256

peta = {f.name: f for pola in ("*_pra.png", "*_pasca.png", "*_gabung.png")
        for f in BASIS.rglob(pola)}
sampel = pd.read_csv(sorted(BASIS.rglob("sampel_kecil.csv"))[0])

CHIP = []
for kelas in ("no-damage", "major-damage", "destroyed"):
    for _, r in sampel[sampel["kelas_xbd"] == kelas].iterrows():
        g = str(r["chip_pra"]).replace("_pra", "_gabung")
        if all(n in peta for n in (r["chip_pra"], r["chip_pasca"], g)):
            CHIP.append({"uid": r["uid"], "kelas_xbd": kelas, "pra": peta[r["chip_pra"]],
                         "pasca": peta[r["chip_pasca"]], "gabung": peta[g]})
            break

assert len(CHIP) == 3, f"hanya dapat {len(CHIP)} chip, butuh 3"
for c in CHIP:
    print(f"  {c['uid'][:8]}  {c['kelas_xbd']}")

from PIL import Image
for c in CHIP:
    print("\n---", c["kelas_xbd"], c["uid"][:8], "---")
    display(Image.open(c["gabung"]))

## Sel 4 — empat bentuk pertanyaan

Keempatnya sengaja tidak menyediakan jalan keluar `tidak_jelas`.
`P2` dan `P3` bahkan memaksa jawaban satu kata, sehingga model tidak bisa mengambang.

| Kode | Masukan | Skala | Pertanyaan |
|---|---|---|---|
| P1 | chip gabung | asli | deskripsi bebas dua sampai tiga kalimat |
| P2 | chip gabung | asli | satu kata: SAMA atau BERUBAH |
| P3 | chip pasca | asli (448) | satu kata: UTUH atau HANCUR |
| P4 | chip pasca | **2x (896)** | satu kata: UTUH atau HANCUR — uji pengaruh perbesaran |

In [ ]:
PROBE = {
 "P1": ("gabung", 1.0,
        "Citra ini memuat dua panel berdampingan: panel kiri diambil sebelum bencana, "
        "panel kanan sesudah bencana. Perhatikan bangunan yang ditunjuk bidik silang "
        "magenta di tengah. Jelaskan dalam dua sampai tiga kalimat bahasa Indonesia apa "
        "yang Anda lihat pada bangunan itu di panel kanan."),
 "P2": ("gabung", 1.0,
        "Citra ini memuat dua panel berdampingan: kiri sebelum bencana, kanan sesudah "
        "bencana. Perhatikan bangunan di dalam kotak magenta. Apakah bangunan itu SAMA "
        "atau BERUBAH? Jawab hanya dengan satu kata: SAMA atau BERUBAH."),
 "P3": ("pasca", 1.0,
        "Perhatikan bangunan yang ditunjuk bidik silang magenta di tengah citra ini. "
        "Apakah bangunan itu UTUH atau HANCUR? Jawab hanya dengan satu kata: "
        "UTUH atau HANCUR."),
 "P4": ("pasca", 2.0,
        "Perhatikan bangunan yang ditunjuk bidik silang magenta di tengah citra ini. "
        "Apakah bangunan itu UTUH atau HANCUR? Jawab hanya dengan satu kata: "
        "UTUH atau HANCUR."),
}
for k, (sumber, skala, teks) in PROBE.items():
    print(f"{k}  sumber={sumber:<7} skala={skala}x  {teks[:60]}...")

## Sel 5 — muat Qwen2.5-VL-3B

In [ ]:
import torch
from PIL import Image
from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor

def _muat(kelas, nama, **kw):
    try:
        return kelas.from_pretrained(nama, dtype=torch.float16, **kw)
    except TypeError:
        return kelas.from_pretrained(nama, torch_dtype=torch.float16, **kw)

# Kaggle kerap memberi dua T4. device_map="auto" akan memecah model ke dua kartu,
# sedangkan InternVL di sel 6 memanggil .cuda() yang selalu menunjuk kartu 0.
# Keduanya dikunci ke kartu 0 supaya tidak ada tensor yang berbeda perangkat.
qwen = _muat(Qwen2_5_VLForConditionalGeneration, MODEL_QWEN, device_map={"": 0}).eval()
qproc = AutoProcessor.from_pretrained(MODEL_QWEN)
print("Qwen siap")

def sajikan(jalur, skala):
    # Skala 1.0 berarti ukuran asli chip. Chip tidak pernah diperkecil di sini,
    # karena memperkecil justru membuang detail yang sedang diuji keterbacaannya.
    im = Image.open(jalur).convert("RGB")
    if skala and skala != 1.0:
        w, h = im.size
        im = im.resize((round(w * skala), round(h * skala)), Image.LANCZOS)
    return im

def jalan_qwen(gambar, teks):
    isi = [{"type": "image", "image": g} for g in gambar] + [{"type": "text", "text": teks}]
    chat = qproc.apply_chat_template([{"role": "user", "content": isi}],
                                     tokenize=False, add_generation_prompt=True)
    masuk = qproc(text=[chat], images=gambar, return_tensors="pt", padding=True).to(qwen.device)
    with torch.inference_mode():
        keluar = qwen.generate(**masuk, max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    return qproc.decode(keluar[0][masuk["input_ids"].shape[1]:], skip_special_tokens=True)

## Sel 6 — muat InternVL3-2B

In [ ]:
import torch, torchvision.transforms as T
from PIL import Image
from torchvision.transforms.functional import InterpolationMode
from transformers import AutoModel, AutoTokenizer

RERATA, SIMPANGAN = (0.485, 0.456, 0.406), (0.229, 0.224, 0.225)

def _ubah(n=448):
    return T.Compose([T.Lambda(lambda im: im.convert("RGB")),
                      T.Resize((n, n), interpolation=InterpolationMode.BICUBIC),
                      T.ToTensor(), T.Normalize(RERATA, SIMPANGAN)])

def _rasio_terdekat(ar, daftar, w, h, n):
    terbaik, beda = (1, 1), float("inf")
    for r in daftar:
        d = abs(ar - r[0] / r[1])
        if d < beda or (d == beda and w * h > 0.5 * n * n * r[0] * r[1]):
            beda, terbaik = d, r
    return terbaik

def _petak(im, n=448, maks=6, thumbnail=True):
    w, h = im.size
    daftar = sorted({(i, j) for k in range(1, maks + 1)
                     for i in range(1, k + 1) for j in range(1, k + 1)
                     if 1 <= i * j <= maks}, key=lambda x: x[0] * x[1])
    ri, rj = _rasio_terdekat(w / h, daftar, w, h, n)
    ubah = im.resize((n * ri, n * rj))
    hasil = [ubah.crop(((i % ri) * n, (i // ri) * n, ((i % ri) + 1) * n, ((i // ri) + 1) * n))
             for i in range(ri * rj)]
    if thumbnail and len(hasil) != 1:
        hasil.append(im.resize((n, n)))
    return hasil

def piksel_internvl(im, maks):
    tf = _ubah(448)
    return torch.stack([tf(p) for p in _petak(im, 448, maks)])

INTERNVL_SIAP = False
def _muat_ivl():
    return AutoModel.from_pretrained(MODEL_INTERNVL, torch_dtype=torch.bfloat16,
                                     low_cpu_mem_usage=True, use_flash_attn=False,
                                     trust_remote_code=True).eval().cuda()
try:
    try:
        ivl = _muat_ivl()
    except AttributeError as e:
        if "all_tied_weights_keys" not in str(e):
            raise
        import transformers
        transformers.modeling_utils.PreTrainedModel.all_tied_weights_keys = property(
            lambda self: list(getattr(self, "_tied_weights_keys", None) or []))
        ivl = _muat_ivl()
    itok = AutoTokenizer.from_pretrained(MODEL_INTERNVL, trust_remote_code=True, use_fast=False)
    GEN = dict(max_new_tokens=MAX_NEW_TOKENS, do_sample=False)
    INTERNVL_SIAP = True
    print("InternVL siap")
except Exception as e:
    print(">>> InternVL gagal dimuat, uji tetap lanjut dengan Qwen saja.")
    print(">>> ", type(e).__name__, str(e)[:300])

def jalan_internvl(gambar, teks, maks_petak):
    px = [piksel_internvl(g, maks_petak).to(torch.bfloat16).cuda() for g in gambar]
    return ivl.chat(itok, torch.cat(px, dim=0), "<image>\n" + teks, GEN,
                    num_patches_list=[p.size(0) for p in px])

## Sel 7 — jalankan uji kendali

3 chip × 4 pertanyaan × 2 model = 24 panggilan, sekitar 4–6 menit.

In [ ]:
import json, time, traceback

HASIL = []
for nama_model in (["qwen2.5-vl-3b"] + (["internvl3-2b"] if INTERNVL_SIAP else [])):
    for kode, (sumber, skala, teks) in PROBE.items():
        for c in CHIP:
            im = sajikan(c[sumber], skala)
            t0 = time.time()
            try:
                if nama_model.startswith("qwen"):
                    jawab = jalan_qwen([im], teks)
                else:
                    jawab = jalan_internvl([im], teks, 6 if sumber == "gabung" else 4)
                galat = None
            except Exception:
                jawab, galat = None, traceback.format_exc(limit=3)
            HASIL.append({"model": nama_model, "probe": kode, "sumber": sumber,
                          "skala": skala, "ukuran_px": list(im.size),
                          "uid": c["uid"], "kelas_xbd": c["kelas_xbd"],
                          "detik": round(time.time() - t0, 2),
                          "keluaran_mentah": jawab, "galat": galat})
            print(("OK " if galat is None else "!! ") +
                  f"{nama_model:<15}{kode}  {c['kelas_xbd']:<13}{HASIL[-1]['detik']:>6.2f}s")
print("\nselesai:", len(HASIL), "panggilan")

## Sel 8 — baca hasilnya

Jawaban dikelompokkan per pertanyaan agar ketiga kelas berdampingan. Perhatikan apakah
baris `no-damage` dan `destroyed` berbeda.

In [ ]:
import json

urut = {"no-damage": 0, "major-damage": 1, "destroyed": 2}
for m in sorted({r["model"] for r in HASIL}):
    for kode in PROBE:
        sub = sorted([r for r in HASIL if r["model"] == m and r["probe"] == kode],
                     key=lambda r: urut[r["kelas_xbd"]])
        print("=" * 78)
        print(f"{m}  {kode}  ({PROBE[kode][0]} @ skala {PROBE[kode][1]}x"
              f" = {sub[0]['ukuran_px'] if sub else '?'})")
        print("-" * 78)
        for r in sub:
            teks = (r["galat"] or r["keluaran_mentah"] or "").strip().replace("\n", " ")
            print(f"  {r['kelas_xbd']:<14} -> {teks[:200]}")
        unik = len({(r["keluaran_mentah"] or "").strip() for r in sub})
        print(f"  [jawaban unik: {unik}/3]")

# /kaggle/input bersifat read-only, jadi keluaran ditulis ke /kaggle/working
KELUARAN = KERJA / "uji_kendali_mentah.jsonl"
with open(KELUARAN, "w", encoding="utf-8") as fh:
    for r in HASIL:
        fh.write(json.dumps(r, ensure_ascii=False) + "\n")
print("\ntersimpan:", KELUARAN, f"({KELUARAN.stat().st_size/1024:.1f} KB)")
print("Ambil berkasnya di panel Output sebelah kanan, atau File > Download.")
print("Catatan: isi /kaggle/working hilang kalau sesi dihentikan tanpa Save Version.")